In [1]:
# Install required packages
!pip install paho-mqtt

# Import libraries
import paho.mqtt.client as mqtt
import json
import time
from datetime import datetime

# Clear output for cleaner display (optional in Jupyter/Kaggle)
from IPython.display import clear_output

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 2.6 MB/s eta 0:00:00


In [2]:
# Callback when subscriber connects to the broker
def on_connect(client, userdata, flags, reason_code, properties):
    print(f"Subscriber connected to broker with code {reason_code}")
    # Subscribe to both topics from the main notebook
    client.subscribe("modulation/train")
    client.subscribe("modulation/predict")

# Callback when a message is received
def on_message(client, userdata, msg):
    try:
        # Decode the JSON payload
        payload = json.loads(msg.payload.decode('utf-8'))
        topic = msg.topic
        
        # Clear previous output for a clean display (optional)
        clear_output(wait=True)
        
        # Process messages based on topic
        if topic == "modulation/train":
            print("\n--- Training Update ---")
            print(f"Epoch: {payload['epoch']}")
            print(f"Training Accuracy: {payload['accuracy']:.4f}")
            print(f"Validation Accuracy: {payload['val_accuracy']:.4f}")
            print(f"Allocated Channel: {payload['allocated_channel']}")
            print(f"Spectrum Stats:")
            print(f"  Occupied Channels: {payload['spectrum_stats']['occupied_channels']}")
            print(f"  Free Channels: {payload['spectrum_stats']['free_channels']}")
            print(f"  Average Quality: {payload['spectrum_stats']['average_quality']:.4f}")
        
        elif topic == "modulation/predict":
            print("\n--- Prediction Update ---")
            print(f"Predicted Class: {payload['predicted']}")
            print(f"True Class: {payload['true']}")
            print(f"Confidence: {payload['confidence']:.4f}")
            print(f"Allocated Channel: {payload['allocated_channel']}")
            print(f"Spectrum Stats:")
            print(f"  Occupied Channels: {payload['spectrum_stats']['occupied_channels']}")
            print(f"  Free Channels: {payload['spectrum_stats']['free_channels']}")
            print(f"  Average Quality: {payload['spectrum_stats']['average_quality']:.4f}")
            print(f"Timestamp: {datetime.fromtimestamp(payload['time']).strftime('%Y-%m-%d %H:%M:%S')}")
    
    except Exception as e:
        print(f"Error processing message: {e}")

In [ ]:
# Configure MQTT subscriber
broker = "test.mosquitto.org"  # Change to "test.mosquitto.org" or your broker if needed
port = 1883
client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2)

# Assign callbacks
client.on_connect = on_connect
client.on_message = on_message

# Connect to broker and start loop
print("Connecting to MQTT broker...")
client.connect(broker, port)
client.loop_start()

# Keep the subscriber running (adjust time or use a loop as needed)
try:
    while True:
        time.sleep(1)  # Keep the notebook running to receive messages
except KeyboardInterrupt:
    print("\nStopping subscriber...")
    client.loop_stop()
    client.disconnect()


--- Training Update ---
Epoch: 26
Training Accuracy: 0.8298
Validation Accuracy: 0.8267
Allocated Channel: 12
Spectrum Stats:
  Occupied Channels: 45
  Free Channels: 19
  Average Quality: 0.3569
